In [2]:
# ======================================================
# IMPORT LIBRARY
# ======================================================

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from imblearn.over_sampling import ADASYN

import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# ======================================================
# LOAD DATA
# ======================================================

data_path = Path("data") / "matches.xlsx"
df = pd.read_excel(data_path)

print("Shape data awal:", df.shape)

Shape data awal: (19847, 88)


In [4]:
# ======================================================
# PILIH KOLOM
# ======================================================

cols = [
    'match_date_start',
    'match_duration',
    'match_tournament',
    'match_premier_status',
    'match_age_rating',
    'match_content_type',
    'match_coverage',
    'match_genre',
    'match_main_genre',
    'match_channel',
    'match_gender',
    'match_organization',
    'match_priority_level'
]

df = df[cols].copy()

In [5]:
# ======================================================
# FEATURE ENGINEERING
# ======================================================

df["match_date_start"] = pd.to_datetime(df["match_date_start"])

df["match_hour"] = df["match_date_start"].dt.hour
df["match_day"] = df["match_date_start"].dt.dayofweek
df["match_month"] = df["match_date_start"].dt.month

df["is_weekend"] = df["match_day"].isin([5,6]).astype(int)
df["is_prime_time"] = df["match_hour"].between(18,23).astype(int)

df.drop(columns=["match_date_start"], inplace=True)

In [6]:
# ======================================================
# CLEAN TARGET
# ======================================================

df["match_priority_level"] = (
    df["match_priority_level"]
    .astype(str)
    .str.strip()
    .str.lower()
)

priority_map = {
    "low":0,
    "medium":1,
    "high":2
}

df["match_priority_level"] = df["match_priority_level"].map(priority_map)

df = df.dropna(subset=["match_priority_level"])

print("\nDistribusi target:")
print(df["match_priority_level"].value_counts())


Distribusi target:
match_priority_level
1.0    17647
2.0     1422
0.0      722
Name: count, dtype: int64


In [7]:
# ======================================================
# ENCODING CATEGORICAL
# ======================================================

cat_cols = df.select_dtypes(include="object").columns

for col in cat_cols:
    
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

In [8]:
# ======================================================
# SPLIT FEATURE & TARGET
# ======================================================

X = df.drop(columns=["match_priority_level"])
y = df["match_priority_level"]

In [9]:
# ======================================================
# TRAIN TEST SPLIT
# ======================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)



Train shape: (15832, 16)
Test shape: (3959, 16)


In [10]:
# ======================================================
# HANDLE IMBALANCED DATA (ADASYN)
# ======================================================

adasyn = ADASYN(
    sampling_strategy="not majority",
    random_state=42,
    n_neighbors=5
)

X_train_res, y_train_res = adasyn.fit_resample(
    X_train,
    y_train
)

print("\nDistribusi setelah ADASYN:")
print(pd.Series(y_train_res).value_counts())


Distribusi setelah ADASYN:
match_priority_level
1.0    14117
0.0    14083
2.0    14033
Name: count, dtype: int64


In [12]:
# ======================================================
# RANDOM FOREST MODEL
# ======================================================

rf_model = RandomForestClassifier(
    n_estimators=400,
    max_depth=12,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_res, y_train_res)

rf_pred = rf_model.predict(X_test)

print("\n===== RANDOM FOREST =====")

print("Accuracy:", accuracy_score(y_test, rf_pred))
print(classification_report(y_test, rf_pred))


===== RANDOM FOREST =====
Accuracy: 0.8423844405152816
              precision    recall  f1-score   support

         0.0       0.55      0.92      0.69       144
         1.0       0.98      0.84      0.90      3530
         2.0       0.35      0.87      0.50       285

    accuracy                           0.84      3959
   macro avg       0.63      0.87      0.70      3959
weighted avg       0.92      0.84      0.87      3959



In [13]:
# ======================================================
# XGBOOST MODEL
# ======================================================

xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42
)

xgb_model.fit(X_train_res, y_train_res)

xgb_pred = xgb_model.predict(X_test)

print("\n===== XGBOOST =====")

print("Accuracy:", accuracy_score(y_test, xgb_pred))
print(classification_report(y_test, xgb_pred))


===== XGBOOST =====
Accuracy: 0.9037635766607729
              precision    recall  f1-score   support

         0.0       0.63      0.89      0.74       144
         1.0       0.98      0.91      0.94      3530
         2.0       0.49      0.83      0.62       285

    accuracy                           0.90      3959
   macro avg       0.70      0.88      0.77      3959
weighted avg       0.93      0.90      0.91      3959

